# 06 · Suitability and validation (Phase 7): MCDA comparison and sensitivity

Abhinav's part of P7.7:

1. the MCDA / AHP baseline: weights, consistency and how each criterion does on its own (P7.3);
2. MCDA next to the learned model and the distance-to-built baseline (P7.5 preview; the harness in P7.4 adds TOC, lift and the LEI split);
3. sensitivity to the slope threshold, k and cell size (P7.6).

*Harsh's part (temporal validation of the combined score, suitability classes, candidate sites) is added with `validate.py` / `score.py`.*

Build: `python -m src.suitability.mcda`, `python -m src.suitability.sensitivity`, `python -m src.viz.charts`. Checked by `scripts/verify_mcda.py` and `scripts/verify_sensitivity.py`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from src.config import load_config

cfg = load_config()
MET = Path(cfg.paths["metrics"])
FIG = Path(cfg.paths["figures"])


def show(name, width=760):
    display(Image(filename=str(FIG / f"{name}.png"), width=width))

## 1. MCDA / AHP

In [ ]:
from src.suitability import mcda

A = pd.DataFrame(mcda.parse_matrix(cfg["mcda"]["ahp"]), index=cfg["mcda"]["criteria"], columns=cfg["mcda"]["criteria"])
w, cons = mcda.weights(cfg)
display(A.round(2))
print("weights:", w.round(3).to_dict())
print(f"lambda_max {cons['lambda_max']:.3f}, consistency ratio {cons['consistency_ratio']:.4f} (< 0.1)")

In [ ]:
m = json.loads((MET / "mcda.json").read_text())
pd.Series(m["criterion_auc"]).round(3).to_frame("AUC on grew, criterion alone").sort_values(by="AUC on grew, criterion alone", ascending=False)

Only distance to built-up carries real signal on its own (0.763); roads, water and land cover are close to chance. The AHP weights spread importance over all five, which dilutes it: the overlay scores 0.636. The weights are **not** tuned on the test label; this is what a fixed expert overlay does.

## 2. Comparison

In [ ]:
sim = json.loads((MET / "similarity.json").read_text())
pd.DataFrame(
    {
        "AUC on grew (validation run)": {
            "change RF (out of fold)": sim["rf_change"]["temporal_preview"]["auc_grew_oof"],
            "distance to built-up": m["baseline_dist_to_built_auc"],
            "state RF (out of fold)": sim["rf_state"]["temporal_preview"]["auc_grew_oof"],
            "MCDA / AHP": m["mcda"]["temporal_preview"]["auc_grew"],
            "MCDA without land cover": m["mcda_no_lulc"]["temporal_preview"]["auc_grew"],
            "random": 0.5,
        }
    }
).round(3)

In [ ]:
show("chart_roc", 520)
show("map_score_mcda_final", 600)

## 3. Sensitivity

In [ ]:
s = json.loads((MET / "sensitivity.json").read_text())
cols = ["n_candidates", "n_grew", "auc_rf_change", "auc_dist_to_built", "auc_mcda"]
display(pd.DataFrame(s["slope"]).set_index("slope_max_deg")[["n_excl_slope", *cols]].round(3))
display(pd.DataFrame(s["cell"]).set_index("cell_size_m")[["n_cells", *cols]].round(3))
display(pd.DataFrame(s["k"]).set_index("k").drop(columns="cluster_rules").round(3))

In [ ]:
show("chart_sensitivity")

**Reading it.** The slope threshold and k barely move anything (AUC within 0.01; ESRI agreement 96.6–98.8 %). Across cell sizes the AUCs aren't comparable, because the growth label itself changes with the cell (and 200 m leaves only 148 grown cells). What holds everywhere is the ranking: change RF > distance to built-up > MCDA.